In [48]:
pip install -i https://mirror-pypi.runflare.com/simple scikit-learn


Looking in indexes: https://mirror-pypi.runflare.com/simple
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 1.3 MB/s  0:00:07 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [41]:
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelBinarizer

from keras.models import Sequential
from keras.layers import Dense, Input
from keras.losses import MeanAbsolutePercentageError
from keras.optimizers import Adam

In [2]:
txt_data_path = "https://raw.githubusercontent.com/emanhamed/Houses-dataset/master/Houses%20Dataset/HousesInfo.txt"

culomns_name = ["bedrooms", "bathrooms", "area", "zipcode", "price"]
df = pd.read_csv(txt_data_path, sep=" ", header=None, names=culomns_name)
df.head(10)

,bedrooms,bathrooms,area,zipcode,price
0,4,4.0,4053,85255,869500
1,4,3.0,3343,36372,865200
2,3,4.0,3923,85266,889000
3,5,5.0,4022,85262,910000
4,3,4.0,4116,85266,971226
5,4,5.0,4581,85266,1249000
6,3,4.0,2544,85262,799000
7,4,5.0,5524,85266,1698000
8,3,4.0,4229,85255,1749000
9,4,5.0,3550,85262,1500000


In [3]:
df.tail(10)

,bedrooms,bathrooms,area,zipcode,price
525,5,3.0,3679,94531,525000
526,4,2.5,1794,94531,380000
527,4,3.5,3420,94531,550000
528,4,3.0,2506,94531,449950
529,4,2.5,2236,94531,495000
530,5,2.0,2066,94531,399900
531,4,3.5,9536,94531,460000
532,3,2.0,2014,94531,407000
533,4,3.0,2312,94531,419000
534,5,3.0,3796,94531,615000


In [4]:
counts = df["zipcode"].value_counts()
type(counts)
unique_zipcodes = counts.index.to_numpy()
counts_zipcodes = counts.values

print(counts_zipcodes.shape)

(49,)


In [5]:
for zipcode, count in zip(unique_zipcodes, counts_zipcodes):
    if count >= 10:
        print(f"{zipcode} --> {count}")

92276 --> 100
93510 --> 60
93446 --> 54
92880 --> 49
94501 --> 41
91901 --> 32
92677 --> 26
94531 --> 22
85255 --> 12
96019 --> 12
85266 --> 11
81524 --> 11
92021 --> 11
93111 --> 11
95220 --> 10


In [6]:
list_least_zipcode = counts[counts.values < 25].index.to_list()

for zipcode in list_least_zipcode:
    idxs = df[df['zipcode'] == zipcode].index
    df.drop(idxs, inplace=True)
    

In [7]:
df

,bedrooms,bathrooms,area,zipcode,price
30,5,3.0,2520,93446,789000
32,3,2.0,1802,93446,365000
39,3,3.0,2146,93446,455000
80,4,2.5,2464,91901,599000
81,2,2.0,1845,91901,529800
...,...,...,...,...,...
499,4,4.0,3000,93446,1495000
500,3,2.0,2330,93446,599900
501,3,2.5,1339,93446,344900
502,3,2.0,1472,93446,309995


In [8]:
df["zipcode"].value_counts()

zipcode
92276    100
93510     60
93446     54
92880     49
94501     41
91901     32
92677     26
Name: count, dtype: int64

In [9]:
train, val = train_test_split(df, test_size=0.2, random_state=42)

In [11]:
train.shape

(289, 5)

In [12]:
val.shape

(73, 5)

In [ ]:
# Normalaize Y based train data
MaxTrain = train["price"].max()

y_train = train["price"] / MaxTrain
y_val = val["price"] / MaxTrain

In [19]:
sc = MinMaxScaler()


In [29]:
train_data_continues = sc.fit_transform(train[['bedrooms', 'bathrooms', 'area']])
val_data_continues = sc.fit_transform(val[['bedrooms', 'bathrooms', 'area']])

In [28]:
binarizer = LabelBinarizer().fit(df['zipcode'])
train_data_categorical = binarizer.transform(train['zipcode'])
val_data_categorical = binarizer.transform(val['zipcode'])

train_data_categorical.shape

(289, 7)

In [31]:
train_data = np.hstack((train_data_continues, train_data_categorical))
val_data = np.hstack((val_data_continues, val_data_categorical))

print(f"Number of features id train data is: {train_data.shape[1]}")
print(f"Number of features id val data is: {val_data.shape[1]}")

Number of features id train data is: 10
Number of features id val data is: 10


In [53]:
model = Sequential()

model.add(Input(shape=(train_data.shape[1], )))
model.add(Dense(8, activation='relu'))
model.add(Dense(4, activation='relu'))
model.add(Dense(1, activation='linear'))

In [54]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 8)              │            88 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 129 (516.00 B)

 Trainable params: 129 (516.00 B)

 Non-trainable params: 0 (0.00 B)

In [55]:
model.compile(optimizer=Adam(learning_rate=0.001, weight_decay=1e-4),
              loss=MeanAbsolutePercentageError())

In [56]:
model.fit(train_data, y_train, validation_data=(val_data, y_val), batch_size=8, epochs=200)

Epoch 1/200
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 886.4706 - val_loss: 509.6492
Epoch 2/200
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 345.3121 - val_loss: 143.0024
Epoch 3/200
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 58.6489 - val_loss: 98.5115
Epoch 4/200
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 42.0246 - val_loss: 83.5414
Epoch 5/200
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 41.7203 - val_loss: 90.4052
Epoch 6/200
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 40.3338 - val_loss: 79.4443
Epoch 7/200
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 37.5115 - val_loss: 79.3554
Epoch 8/200
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 37.6460 - val_loss: 82.7304
Epoch 9/200
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 36.1643 - val_loss: 79.2864
Epoch 10/200
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 35.7788 - val_loss: 77.3476
Epoch 11/200
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 36.1298 - val_loss: 74.1900
Epoch 12/200
37/37 ━━━━━━━━━━━━━━━━━━